In [ ]:
import pandas as pd

path = "data/av2_mf_tiny/train/0000b0f9-99f9-4a1f-a231-5be9e4c523f7/scenario_0000b0f9-99f9-4a1f-a231-5be9e4c523f7.parquet"
df = pd.read_parquet(path)   # 需要 pyarrow
print(df.shape)
print(df.columns)
print(df.head())

(2511, 16)
Index(['observed', 'track_id', 'object_type', 'object_category', 'timestep',
       'position_x', 'position_y', 'heading', 'velocity_x', 'velocity_y',
       'scenario_id', 'start_timestamp', 'end_timestamp', 'num_timestamps',
       'focal_track_id', 'city'],
      dtype='object')
   observed track_id object_type  object_category  timestep  position_x  \
0      True    50513     vehicle                2         0   18.591604   
1      True    50513     vehicle                2         1   18.693685   
2      True    50513     vehicle                2         2   18.819566   
3      True    50513     vehicle                2         3   18.967646   
4      True    50513     vehicle                2         4   19.136512   

   position_y   heading  velocity_x  velocity_y  \
0 -414.075877  1.269081    1.889325    6.052013   
1 -413.747185  1.268987    1.898718    6.062060   
2 -413.338909  1.268902    1.902845    6.048986   
3 -412.857532  1.268811    1.890446    5.996844   


In [2]:
import pandas as pd
import numpy as np
import random
import time
import wandb
import torch
from datasets.Dataset import Dataset
from datasets.InD import InD
from datasets.EthUcy import EthUcy
from model.TrajFlow import TrajFlow, CausalEnocder, Flow
from train import train
from evaluate import evaluate
from visualize import visualize
from visualize_temp import visualize_temp

def load_pz_t1(csv_path, device="cpu", dtype=torch.float32):
    df = pd.read_csv(csv_path)          # shape: (steps*steps, T)
    arr = df.values                     # NumPy array
    pz_t1 = torch.tensor(arr, dtype=dtype, device=device)
    return pz_t1

import matplotlib.pyplot as plt
import numpy as np

def save_heatmap(likelihood, t, output_path, cmap="viridis"):
    plt.figure(figsize=(6,5))
    plt.imshow(likelihood, cmap=cmap, origin="lower")
    plt.colorbar(label="Likelihood")
    plt.title(f"Heatmap at t={t}")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


pz_t1 = load_pz_t1(f"pz_t1_{5}.csv")
steps = 1000
for t in range(100):
    likelihood = pz_t1[:, t].cpu().numpy().reshape(steps, steps)
    likelihood = likelihood / np.max(likelihood)
    df = pd.DataFrame(likelihood)
    df.to_csv(f"likelihood_{5}_{t}.csv", index=False)


    # Save heatmap
    save_heatmap(likelihood, t, f"heatmap_5_{t}.png")



In [3]:
df

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,4.537533e-18,4.001701e-18,3.521897e-18,3.093313e-18,2.711056e-18,2.497066e-18,2.348480e-18,2.259071e-18,2.170208e-18,2.083306e-18,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
996,3.028639e-18,2.665586e-18,2.341252e-18,2.052197e-18,1.794916e-18,1.657286e-18,1.558682e-18,1.497747e-18,1.436726e-18,1.377969e-18,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
997,2.015800e-18,1.770671e-18,1.552026e-18,1.357766e-18,1.185087e-18,1.096879e-18,1.031705e-18,9.902299e-19,9.485568e-19,9.089287e-19,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
998,1.337889e-18,1.172845e-18,1.025940e-18,8.956260e-19,7.802259e-19,7.239317e-19,6.809379e-19,6.529011e-19,6.244587e-19,5.978458e-19,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
pd.read_csv("pz_t1_0.csv")

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
999996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
999997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
999998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
grids = pd.read_csv("denormalized_grid_1.csv")
heatmaps = pd.read_csv("pz_t1_0.csv")

df_av = pd.read_csv("unobserved_traj_1.csv")



grids

,x,y
0,25.0,-65.000000
1,25.0,-64.944945
2,25.0,-64.889890
3,25.0,-64.834835
4,25.0,-64.779780
...,...,...
999995,85.0,-10.220220
999996,85.0,-10.165165
999997,85.0,-10.110110
999998,85.0,-10.055055


In [6]:
df_av

,x,y
0,53.178129,23.132209
1,53.050849,23.103731
2,52.922300,23.074211
3,52.792020,23.042821
4,52.660400,23.009701
...,...,...
95,38.071340,11.399589
96,37.860940,11.181968
97,37.647590,10.961040
98,37.431550,10.737201


In [7]:
# Load grid coordinates and plot a canvas you can overlay trajectories on later
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

df = pd.read_csv("denormalized_grid_1.csv")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) >= 2:
    xcol, ycol = numeric_cols[0], numeric_cols[1]
else:
    raise ValueError("Need two numeric columns.")


df1= pd.read_csv("unobserved_traj_5.csv")

df1_known = pd.read_csv("observed_traj_5.csv")
df_av = pd.read_csv("unobserved_traj_2.csv")
df_av_known = pd.read_csv("observed_traj_2.csv")

df["position_x"] = df["x"] 
df["position_y"] = df["y"] 

df_av["position_x"] = df_av["x"] 
df_av["position_y"] = -df_av["y"] 
df1["position_x"] = df1["x"]
df1["position_y"] = -df1["y"]

df_av_known["position_x"] = df_av_known["x"] 
df_av_known["position_y"] = -df_av_known["y"] 
df1_known["position_x"] = df1_known["x"]
df1_known["position_y"] = -df1_known["y"]

# Plot only
plt.figure(figsize=(8,8))
plt.scatter(df["position_x"], df["position_y"], s=0.1)
plt.plot(df_av["position_x"], df_av["position_y"], color = "red", label = "AV (2)")
plt.plot(df_av_known["position_x"], df_av_known["position_y"], color = "pink", label = "AV (2) known")
plt.plot(df1_known["position_x"], df1_known["position_y"], color = "green", label = "Agent (5)_known")
plt.plot(df1["position_x"], df1["position_y"], color = "purple", label = "Agent (5)")
plt.gca().set_aspect('equal', 'box')
plt.xlabel(xcol)
plt.ylabel(ycol)
plt.title("Grid Canvas")
plt.tight_layout()
plt.legend()


plt.savefig("grids.png", dpi=300)



/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91628/3605982648.py:50: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig("grids.png", dpi=300)


In [8]:
#Data_processing
# df_av["position_x"] = df_av["x"] 
# df_av["position_y"] = -df_av["y"] 
# df1["position_x"] = df1["x"]
# df1["position_y"] = -df1["y"]


delta_t = 0.04  # seconds between frames

# ----- Compute velocities for df_av -----
dx = df_av["position_x"].diff()
dy = df_av["position_y"].diff()

df_av["velocity_x"] = dx / delta_t
df_av["velocity_y"] = dy / delta_t

# df_av["velocity_x"].iloc[0] = 0.0
# df_av["velocity_y"].iloc[0] = 0.0

df_av = df_av.iloc[1:].reset_index(drop=True)

# ----- Compute velocities for df1 -----
dx = df1["position_x"].diff()
dy = df1["position_y"].diff()

df1["velocity_x"] = dx / delta_t
df1["velocity_y"] = dy / delta_t

# df1["velocity_x"].iloc[0] = 0.0
# df1["velocity_y"].iloc[0] = 0.0

df1 = df1.iloc[1:].reset_index(drop=True)


In [9]:
# Compute Collision time

# normal vehicle = 5m by 2m: safety box: 10m by 7m
lengths = 5
widths = 2


def calculate_safety_box_av(row):
    car_x = row["position_x"]
    car_y = row["position_y"]
    car_vx = row["velocity_x"]
    car_vy = row["velocity_y"]
    box_length= lengths
    box_width= widths 
    car_angle = np.arctan2(car_vy, car_vx)

    local_corners = np.array([
        [-box_length / 2, -box_width / 2],
        [-box_length / 2, box_width / 2],
        [box_length / 2, box_width / 2],
        [box_length / 2, -box_width / 2],
        [-box_length / 2, -box_width / 2]# Repeat the first point to close the box
    ])
    rotation_matrix = np.array([
        [np.cos(car_angle), -np.sin(car_angle)],
        [np.sin(car_angle), np.cos(car_angle)]
    ])
    global_corners = local_corners.dot(rotation_matrix) + np.array([car_x, car_y])
    global_corners[:, 1] = -global_corners[:, 1] + 2 * car_y
    return global_corners

def calculate_safety_box(row):
    car_x = row["position_x"]
    car_y = row["position_y"]
    car_vx = row["velocity_x"]
    car_vy = row["velocity_y"]
    box_length= lengths
    box_width= widths

    car_angle = np.arctan2(car_vy, car_vx)

    local_corners = np.array([
        [-box_length / 2, -box_width / 2],
        [-box_length / 2, box_width / 2],
        [box_length / 2, box_width / 2],
        [box_length / 2, -box_width / 2],
        [-box_length / 2, -box_width / 2]# Repeat the first point to close the box
    ])
    rotation_matrix = np.array([
        [np.cos(car_angle), -np.sin(car_angle)],
        [np.sin(car_angle), np.cos(car_angle)]
    ])
    global_corners = local_corners.dot(rotation_matrix) + np.array([car_x, car_y])
    global_corners[:, 1] = -global_corners[:, 1] + 2 * car_y
    return global_corners

df_av["corners"] = df_av.apply(lambda x: calculate_safety_box_av(x), axis = 1)
df1["corners"] = df1.apply(lambda x: calculate_safety_box(x), axis = 1)

df_av1 = df_av
df11 = df1

In [10]:
import matplotlib.pyplot as plt

t = 3
plt.plot(df_av.iloc[t]["corners"][:, 0], df_av.iloc[t]["corners"][:, 1], '-o')  # Plot the corners
plt.scatter(df_av.iloc[t]["position_x"], df_av.iloc[t]["position_y"], color='red')  # Mark the car's position
plt.plot(df1.iloc[t]["corners"][:, 0], df1.iloc[t]["corners"][:, 1], '-o')  # Plot the corners
plt.scatter(df1.iloc[t]["position_x"], df1.iloc[t]["position_y"], color='red')  # Mark the car's position

plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.title('Safety Box around the Car')
plt.grid(True)
plt.axis('equal')
plt.savefig("boxes.png", dpi=300)

/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91628/797829322.py:14: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig("boxes.png", dpi=300)


In [11]:
#check global vs focal
from matplotlib.path import Path
import pandas as pd
import numpy as np
import matplotlib.patches as patches
from shapely.geometry import Polygon, Point
import time
import math
from matplotlib import cm


import matplotlib.pyplot as plt
import numpy as np

overall_heatmap = pd.DataFrame()
for t in range(len(df1)):
    lik = pd.read_csv(f"likelihood_5_{t}.csv")
    data_cut = pd.read_csv("denormalized_grid_1.csv")
    # ---- Fix: remove the incorrect first row ----
    #lik = lik.iloc[1:, :]

    # ---- Reshape likelihood to match the grid ----
    lik_long = lik.to_numpy().reshape(-1)

    # ---- Check size matches ----
    if len(data_cut) != len(lik_long):
        raise ValueError(f"Size mismatch: grid={len(data_cut)}, likelihood={len(lik_long)}")

    # ---- Attach probability column ----
    data_cut["score"] = lik_long

    overall_heatmap = pd.concat([overall_heatmap,data_cut])

# overall_heatmap["score2"] = overall_heatmap["score"] - np.min(overall_heatmap["score"].values)



In [16]:
# Load files
lik = pd.read_csv("likelihood_5_3.csv")
data_cut = pd.read_csv("denormalized_grid_1.csv")

# ---- Fix: remove the incorrect first row ----
# lik = lik.iloc[1:, :]

# ---- Reshape likelihood to match the grid ----
lik_long = lik.to_numpy().reshape(-1)

# ---- Check size matches ----
if len(data_cut) != len(lik_long):
    raise ValueError(f"Size mismatch: grid={len(data_cut)}, likelihood={len(lik_long)}")

# ---- Attach probability column ----
data_cut["score"] = lik_long


# Prepare data for heatmap
# data_cut = data[(data["position_x"] >= minx) &(data["position_x"] <= maxx+ 1)& (data["position_y"] >= miny)& (data["position_y"] <= maxy+ 1)]
x = data_cut['x']
y = data_cut['y']
score = data_cut['score']
# Generate a heatmap
plt.figure(figsize=(10, 8))
hb = plt.scatter(x, y, c=score, cmap=cm.RdYlGn_r)
cb = plt.colorbar(hb)
cb.set_label('mean score')

t =3
plt.plot(df_av["position_x"], df_av["position_y"], color = "red", label = "AV (2)")
plt.plot(df_av_known["position_x"], df_av_known["position_y"], color = "pink", label = "AV (2) known")
plt.plot(df1_known["position_x"], df1_known["position_y"], color = "green", label = "Agent (5)_known")
plt.plot(df1["position_x"], df1["position_y"], color = "purple", label = "Agent (5)")

plt.plot(df_av.iloc[t]["corners"][:, 0], df_av.iloc[t]["corners"][:, 1], '-o')  # Plot the corners
plt.scatter(df_av.iloc[t]["position_x"], df_av.iloc[t]["position_y"], color='red')  # Mark the car's position
plt.plot(df1.iloc[t]["corners"][:, 0], df1.iloc[t]["corners"][:, 1], '-o')  # Plot the corners
plt.scatter(df1.iloc[t]["position_x"], df1.iloc[t]["position_y"], color='red')  # Mark the car's position

plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.title('Heatmap based on score value')
plt.tight_layout()
plt.savefig("heatmaps_score.png", dpi=300)



plt.show()


/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91628/3196252402.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [78]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.path import Path

# Hard Coding...
# Assuming df_av and df11 are defined elsewhere in your script
# just for generating the heat_map_test
import time

start_time = time.time()

minx = np.floor(np.min(df_av1["position_x"])-20)-0.05
miny = np.floor(np.min(df_av1["position_y"])-20)-0.05
maxx = np.ceil(np.max(df_av1["position_x"])+20)-0.05
maxy = np.ceil(np.max(df_av1["position_y"])+20)-0.05

print(minx,miny,maxx,maxy)# 2816.0 -1976.0 2844.0 -1953.0

# Define the function for calculating the safety box
def calculate_safety_box(car_x, car_y, car_vx, car_vy, car_angle, box_length, box_width):
    #car_angle = np.arctan2(car_vy, car_vx)
    local_corners = np.array([
        [-box_length / 2, -box_width / 2],
        [-box_length / 2, box_width / 2],
        [box_length / 2, box_width / 2],
        [box_length / 2, -box_width / 2],
        [-box_length / 2, -box_width / 2]  # Repeat the first point to close the box
    ])
    rotation_matrix = np.array([
        [np.cos(car_angle), -np.sin(car_angle)],
        [np.sin(car_angle), np.cos(car_angle)]
    ])
    global_corners = local_corners.dot(rotation_matrix) + np.array([car_x, car_y])
    global_corners[:, 1] = -global_corners[:, 1] + 2 * car_y
    return global_corners

# Optimized function to check if points are inside the box
def is_inside_box_optimized(x_range, y_range, box_corners):
    X, Y = np.meshgrid(x_range, y_range)
    points = np.vstack([X.ravel(), Y.ravel()]).T
    box_path = Path(box_corners)
    grid_inside = box_path.contains_points(points)
    inside_mask = grid_inside.reshape(X.shape)
    return inside_mask.astype(int)

# Function to create the heatmap dataframe
def create_heatmap_dataframe(min_x, max_x, min_y, max_y, box_corners, t, heatmap_data=None):
    x_range = np.arange(min_x, max_x + 1, 0.1)
    y_range = np.arange(min_y, max_y + 1, 0.1)
    x_range = np.round(x_range, 1)
    y_range = np.round(y_range, 1)

    # if t == 0:
    #     inside_mask = is_inside_box_optimized(x_range, y_range, box_corners)
    #     heatmap_df = pd.DataFrame(inside_mask, index=y_range, columns=x_range)
    # else:
    heatmap_df = pd.DataFrame(index=y_range, columns=x_range).fillna(0)
    if heatmap_data is not None:
        heatmap_data_valid = heatmap_data[heatmap_data["prob"] != 0]
        print(len(heatmap_data_valid))
        heatmap_data_valid = heatmap_data_valid[(heatmap_data_valid["position_x"]>= min_x-0.5)&(heatmap_data_valid["position_x"]<= max_x+1.5)&(heatmap_data_valid["position_y"] >= min_y-0.5)&(heatmap_data_valid["position_y"]<= max_y+1.5)]
        print(len(heatmap_data_valid))
        heatmap_data_valid = heatmap_data_valid.sort_values("prob")
        heatmap_data_valid = heatmap_data_valid.reset_index(drop=True)
        for _, row in heatmap_data_valid.iterrows():
            x, y, prob = row["position_x"], row["position_y"], row["prob"]
            # This is important (maybe use of meshgrid or polygon make it faster)
            for x_adj in np.arange(x-0.5,x+0.6,0.1):
                for y_adj in np.arange(y-0.5,y+0.6,0.1):
                    if np.round(y_adj,1) in y_range and np.round(x_adj,1) in x_range:
                        heatmap_df.at[np.round(y_adj,1), np.round(x_adj,1)] = prob

                #if x in x_range and y in y_range:
                #    heatmap_df.at[y, x] = prob
    return heatmap_df

#check global vs focal
from matplotlib.path import Path
import pandas as pd
import numpy as np
import matplotlib.patches as patches
from shapely.geometry import Polygon, Point
import time
import math
from matplotlib import cm


import matplotlib.pyplot as plt
import numpy as np

overall_heatmap = pd.DataFrame()
for t in range(len(df1)):
    lik = pd.read_csv(f"likelihood_5_{t}.csv")
    data_cut = pd.read_csv("denormalized_grid_1.csv")
    # ---- Fix: remove the incorrect first row ----
    #lik = lik.iloc[1:, :]

    # ---- Reshape likelihood to match the grid ----
    lik_long = lik.to_numpy().reshape(-1)

    # ---- Check size matches ----
    if len(data_cut) != len(lik_long):
        raise ValueError(f"Size mismatch: grid={len(data_cut)}, likelihood={len(lik_long)}")

    # ---- Attach probability column ----
    data_cut["score"] = lik_long

    overall_heatmap = pd.concat([overall_heatmap,data_cut])

# overall_heatmap["score2"] = overall_heatmap["score"] - np.min(overall_heatmap["score"].values)


#print(np.min(overall_heatmap["score"].values))
#print(np.max(overall_heatmap["score2"].values))

for t in range(len(df1)):
    lik = pd.read_csv(f"likelihood_5_{t}.csv")
    heatmap_data = pd.read_csv("denormalized_grid_1.csv")
    # ---- Fix: remove the incorrect first row ----
    #lik = lik.iloc[1:, :]

    # ---- Reshape likelihood to match the grid ----
    lik_long = lik.to_numpy().reshape(-1)

    # ---- Check size matches ----
    if len(data_cut) != len(lik_long):
        raise ValueError(f"Size mismatch: grid={len(data_cut)}, likelihood={len(lik_long)}")

    # ---- Attach probability column ----
    heatmap_data["score"] = lik_long

    #heatmap_data["score2"] = heatmap_data["score"] - np.min(heatmap_data["score"].values)
    #heatmap_data["prob"] = heatmap_data["score2"] / np.max(heatmap_data["score2"].values)
    #print(np.min(heatmap_data["score"].values))
    #print(np.max(heatmap_data["score2"].values))

    heatmap_data["score2"] = heatmap_data["score"]# - np.min(overall_heatmap["score"].values)
    heatmap_data["prob"] = heatmap_data["score2"] #/ np.max(overall_heatmap["score2"].values)
    heatmap_data["position_x"] = heatmap_data["x"]
    heatmap_data["position_y"] = heatmap_data["y"]
    #heatmap_data["prob"] = heatmap_data["score"] # if/as it is scaled between 0 and 1

    heatmap_data.to_csv(f"heatmap_test_prob{t}.csv")
        #print(np.max(heatmap_data["prob"]))

    box_length = lengths #lengths[df11.iloc[0]["object_type"]]
    box_width = widths #widths[df11.iloc[0]["object_type"]]


    car_position = (df11.iloc[t]["position_x"], df11.iloc[t]["position_y"])  # x, y
    car_velocity = (df11.iloc[t]["velocity_x"], df11.iloc[t]["velocity_y"])  # vx, vy
    # this is needed for the t=0 heatmap
    car_angle = np.arctan2(df11.iloc[t]["velocity_y"], df11.iloc[t]["velocity_x"])

    overall_heatmap = pd.DataFrame()




    safety_box_corners = calculate_safety_box(*car_position, *car_velocity, car_angle,box_length, box_width)

    #plt.plot(safety_box_corners[:, 0], safety_box_corners[:, 1], '-o')  # Plot the corners
    #plt.scatter(car_position[0], car_position[1], color='red')  # Mark the car's position
    #plt.xlabel('X Coordinate')
    #plt.ylabel('Y Coordinate')
    #plt.title('Safety Box around the Car')
    #plt.grid(True)
    #plt.axis('equal')

    heatmap_df = create_heatmap_dataframe(minx, maxx, miny, maxy, safety_box_corners, t, heatmap_data)
    #heatmap_df.to_csv(f"heatmap_testh_dbg{t}.csv")
    # print(np.max(np.reshape(heatmap_df.to_numpy(),heatmap_df.shape[0]*heatmap_df.shape[1])))
    # print(np.min(np.reshape(heatmap_df.to_numpy(),heatmap_df.shape[0]*heatmap_df.shape[1])))

    heatmap_df.to_csv(f"heatmap_testh{t}.csv")


35.95 -60.05 79.95 -5.05


/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91409/2700788095.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  heatmap_df = pd.DataFrame(index=y_range, columns=x_range).fillna(0)
/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91409/2700788095.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.0000000000000001e-45' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  heatmap_df.at[np.round(y_adj,1), np.round(x_adj,1)] = prob
/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91409/2700788095.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pa

85940
83873


/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91409/2700788095.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.0000000000000001e-45' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  heatmap_df.at[np.round(y_adj,1), np.round(x_adj,1)] = prob
/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91409/2700788095.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.0000000000000001e-45' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  heatmap_df.at[np.round(y_adj,1), np.round(x_adj,1)] = prob
/var/folders/70/7hwsprrj2xb232_my0td8xnr0000gn/T/ipykernel_91409/2700788095.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.0000000000000001e-

KeyboardInterrupt: 

In [15]:
# Preload all CSV files into a dictionary if the dataset is not too large
heatmap_files = [f"heatmap_testh{i}.csv" for i in range(len(df1))]
heatmap_data = {i: pd.read_csv(file) for i, file in enumerate(heatmap_files)}

In [16]:
def load_heatmap(time):
    # Use ceiling for non-integer times, but keep integers as is
    #key = int(math.ceil(time)) if not time.is_integer() else int(time)
    # Limit the key to a maximum of 6
    return heatmap_data[time]

def calculate_safety_box(car_x, car_y, car_vx, car_vy, car_angle, box_length, box_width):
    #car_angle = np.arctan2(car_vy, car_vx)
    local_corners = np.array([
        [-box_length / 2, -box_width / 2],
        [-box_length / 2, box_width / 2],
        [box_length / 2, box_width / 2],
        [box_length / 2, -box_width / 2],
        [-box_length / 2, -box_width / 2]
    ])
    rotation_matrix = np.array([
        [np.cos(car_angle), -np.sin(car_angle)],
        [np.sin(car_angle), np.cos(car_angle)]
    ])
    global_corners = local_corners.dot(rotation_matrix) + np.array([car_x, car_y])
    global_corners[:, 1] = -global_corners[:, 1] + 2 * car_y
    return global_corners

def create_safety_box_grid_dict(corners, resolution_length, resolution_width):
    """ Create a grid of points within the safety box defined by corners and return a dictionary of their global coordinates. """
    top_edge = interpolate_points(corners[0], corners[1], resolution_length)
    bottom_edge = interpolate_points(corners[3], corners[2], resolution_length)

    grid_dict = {}
    for i in range(resolution_length):
        vertical_line = interpolate_points(top_edge[i], bottom_edge[i], resolution_width)
        for j, point in enumerate(vertical_line):
            grid_dict[(i, j)] = point

    return grid_dict

def interpolate_points(start, end, num_points):
    return np.linspace(start, end, num_points, endpoint=True)


def create_safety_box_grid(corners, resolution_length, resolution_width):
    """ Create a grid of points within the safety box defined by corners. """
    top_edge = np.linspace(corners[0], corners[1], resolution_length)
    bottom_edge = np.linspace(corners[3], corners[2], resolution_length)
    grid_x, grid_y = np.meshgrid(top_edge, bottom_edge)
    grid = np.dstack((grid_x, grid_y)).reshape(-1, 2)
    return grid


def create_safety_box_array(corners, resolution_length, resolution_width):
    """ Create an array for the safety box where each cell contains its global coordinates. """
    top_edge = interpolate_points(corners[0], corners[3], resolution_length)
    bottom_edge = interpolate_points(corners[1], corners[2], resolution_length)

    safety_box_array = np.zeros((resolution_width, resolution_length), dtype=object)

    for i in range(resolution_length):
        vertical_line = interpolate_points(top_edge[i], bottom_edge[i], resolution_width)
        for j, point in enumerate(vertical_line):
            safety_box_array[j, i] = point

    return safety_box_array

# need to store the information about the position of the grid in terms of the center location



# get the distance from the point representing the cell and the car position

def create_safety_box_array_distance_to_center(corners, resolution_length, resolution_width,car_x, car_y):
    """ Create an array for the safety box where each cell contains its global coordinates. """
    top_edge = interpolate_points(corners[0], corners[3], resolution_length)
    bottom_edge = interpolate_points(corners[1], corners[2], resolution_length)

    safety_box_array = np.zeros((resolution_width, resolution_length), dtype=object)

    for i in range(resolution_length):
        vertical_line = interpolate_points(top_edge[i], bottom_edge[i], resolution_width)
        for j, point in enumerate(vertical_line):
            safety_box_array[j, i] = np.sqrt((point[0]-car_x)**2+(point[1]-car_y)**2)

    return safety_box_array


def fill_safety_box_with_occupancy(heatmap, car_x, car_y, car_vx, car_vy, car_angle, box_length=30, box_width=28, resolution=1):
    # NEED THE SAFETY BOX DIMENSION BASED ON THE types of vehicles (ego and agent)
    # normal vehicle = 5m by 2m: safety box: 10m by 7m

    #bl = 10
    #bw = 7



    # Dynamically changing safety box - based on the ego's velocity
    # either we can consider the stoppping distance
    # or we can consider the current speed * 0.1s -> m (maximum moving distance of ego vehicle within timestep)
    # I think the second approach make sense.

    # steps of the dynamically changing safety box
    # 1. calculate the velocity in the direction of the heading
    # 2. Then multiply to 0.1s to get the max traveling distance.

    # Lets just use the max velocity of the ego vehicle
    #https://www.omnicalculator.com/physics/stopping-distance
    car_v_max = np.sqrt(car_vx**2+car_vy**2) #m/s
    # dynamic_extension = car_v_max * 0.1  max traveling distance too small

    # s – Stopping distance in meters;
    #t – Perception-reaction time in seconds;
    #v – Speed of the car in km/h;
    #G – Grade (slope) of the road, expressed as a decimal. Positive for an uphill grade and negative for a downhill road; and
    #f – Coefficient of friction between the tires and the road. It is assumed to be 0.7 on a dry road and between 0.3 and 0.4 on a wet road.

    stop_d = (0.278 * 1.5 * 3.6*car_v_max) + (3.6*car_v_max)**2 / (254 * (0.5 + 0))

    # extra = max(lengths[df11.iloc[0]["object_type"]], widths[df11.iloc[0]["object_type"]])
    # bl = lengths[df_av1.iloc[0]["object_type"]]+ extra + stop_d*2
    # bw = widths[df_av1.iloc[0]["object_type"]]+ extra

    extra = max(lengths, widths)
    bl = lengths + extra +stop_d*2 #lengths[df_av1.iloc[0]["object_type"]]+ extra
    bw = lengths + extra #widths[df_av1.iloc[0]["object_type"]]+ extra


    print(bl)
    # adjustment only on the forward

    resolution = 0.1 #0.01

    corners = calculate_safety_box(car_x, car_y, car_vx, car_vy, car_angle, bl, bw)
    resolution_length = math.ceil(bl/resolution)#int(bl/resolution)  # Number of points along the box's length
    resolution_width = math.ceil(bw/resolution) # Number of points along the box's width

    print(resolution_length)

    safety_box_array = create_safety_box_array(corners, resolution_length, resolution_width)
    safety_box_df = pd.DataFrame(safety_box_array)
    print(safety_box_array.shape)

    #safety_box_array_distance = create_safety_box_array_distance_to_center(corners, resolution_length, resolution_width,car_x, car_y)
    #safety_box_df_distance = pd.DataFrame(safety_box_array_distance)

    return corners,safety_box_array, safety_box_df#,safety_box_array_distance,safety_box_df_distance

def fill_safety_box_with_occupancy_og(heatmap, car_x, car_y, car_vx, car_vy, car_angle, box_length=30, box_width=28, resolution=1):
    # NEED THE SAFETY BOX DIMENSION BASED ON THE types of vehicles (ego and agent)
    # normal vehicle = 5m by 2m: safety box: 10m by 7m

    #bl = 10
    #bw = 7


    # original safety box

    #extra = max(lengths[df11.iloc[0]["object_type"]], widths[df11.iloc[0]["object_type"]])
    extra = max(lengths, widths)
    bl = lengths + extra #lengths[df_av1.iloc[0]["object_type"]]+ extra
    bw = lengths + extra #widths[df_av1.iloc[0]["object_type"]]+ extra

    # adjustment only on the forward

    resolution = 0.1 #0.01

    corners = calculate_safety_box(car_x, car_y, car_vx, car_vy, car_angle, bl, bw)
    resolution_length = math.ceil(bl/resolution)#int(bl/resolution)  # Number of points along the box's length
    resolution_width = math.ceil(bw/resolution) # Number of points along the box's width

    safety_box_array = create_safety_box_array(corners, resolution_length, resolution_width)
    safety_box_df = pd.DataFrame(safety_box_array)
    print(safety_box_array.shape)

    #safety_box_array_distance = create_safety_box_array_distance_to_center(corners, resolution_length, resolution_width,car_x, car_y)
    #safety_box_df_distance = pd.DataFrame(safety_box_array_distance)

    return corners,safety_box_array, safety_box_df #,safety_box_array_distance,safety_box_df_distance

# Need to optimize this part
def fill_safety_box_with_heatmap_values(safety_box_array, heatmapf, t):
    t = round(t, 1)
    filled_safety_box = np.zeros_like(safety_box_array, dtype=float)
    heatmap = heatmapf.copy()
    # Assume the first column of heatmap is 'Unnamed: 0', which we will use as the index
    heatmap.index = heatmap["Unnamed: 0"]
    heatmap.drop(columns=["Unnamed: 0"], inplace=True)

    # Convert the column names and index to numeric types for comparison
    heatmap.columns = pd.to_numeric(heatmap.columns)
    heatmap.index = pd.to_numeric(heatmap.index)

    # Iterate through each cell in the safety box grid
    for i in range(safety_box_array.shape[0]):
        for j in range(safety_box_array.shape[1]):
            x, y = safety_box_array[i, j]

            # Check if the point falls within any cell in the heatmap
            x_mask = (heatmap.columns >= x - 0.05) & (heatmap.columns <= x + 0.05)
            y_mask = (heatmap.index >= y - 0.05) & (heatmap.index <= y + 0.05)

            # If the point is within the range, assign the corresponding heatmap value
            #hm_values = heatmap.loc[y_mask, x_mask].values[0]
            #hm_values = list(hm_values)
            #hm_values.append(0)
            #hm_values = np.array(hm_values)
            #filled_safety_box[i, j] = np.max(hm_values)
            if x_mask.any() and y_mask.any():
                hm_values = heatmap.loc[y_mask, x_mask]
                filled_safety_box[i, j] = hm_values.values.max()

    return filled_safety_box


import numpy as np
import pandas as pd
from matplotlib.path import Path

def is_inside_box_optimized(x_range, y_range, box_corners):
    X, Y = np.meshgrid(x_range, y_range)
    points = np.vstack([X.ravel(), Y.ravel()]).T
    box_path = Path(box_corners)
    grid_inside = box_path.contains_points(points)
    inside_mask = grid_inside.reshape(X.shape)
    return inside_mask.astype(int)

def create_heatmap_dataframe(min_x, max_x, min_y, max_y, box_corners):
    # Define the range of coordinates with a step of 1 meter
    x_range = np.arange(min_x, max_x + 1, 0.1)
    y_range = np.arange(min_y, max_y + 1, 0.1)

    # Use the optimized function to check if points are inside the box
    inside_mask = is_inside_box_optimized(x_range, y_range, box_corners)

    # Create a DataFrame from the mask
    heatmap_df = pd.DataFrame(inside_mask, index=np.round(y_range, 1), columns=np.round(x_range, 1))

    heatmap_df

    return heatmap_df

In [17]:

time_series = [] # just the maximum safetybox
time_series_each = [] # dynamical safety box
time_series_og = []

# just for generating the heat_map_test

minx = np.floor(np.min(df_av1["position_x"])-20)-0.05
miny = np.floor(np.min(df_av1["position_y"])-20)-0.05
maxx = np.ceil(np.max(df_av1["position_x"])+20)-0.05
maxy = np.ceil(np.max(df_av1["position_y"])+20)-0.05

print(minx,miny,maxx,maxy)# 2816.0 -1976.0 2844.0 -1953.0

for time in range(len(df1)):
    print(time)
    #print(int(time))
    ego_vehicle_position = (df_av1.iloc[int(time)]["position_x"], df_av1.iloc[int(time)]["position_y"])
    ego_vehicle_velocity = (df_av1.iloc[int(time)]["velocity_x"], df_av1.iloc[int(time)]["velocity_y"])
    ego_vehicle_car_angle = np.arctan2(df_av1.iloc[int(time)]["velocity_y"], df_av1.iloc[int(time)]["velocity_x"])# df_av1.iloc[int(time)]["heading"]
    heatmap = load_heatmap(time)  # Access preloaded heatmap
    #safety_box_corners_large, safety_box_array, safety_box_df,safety_box_array_distance, safety_box_df_distance = fill_safety_box_with_occupancy(heatmap, *ego_vehicle_position, *ego_vehicle_velocity,ego_vehicle_car_angle)

    # original safety box
    safety_box_corners_large_og, safety_box_array_og, safety_box_df_og = fill_safety_box_with_occupancy_og(heatmap, *ego_vehicle_position, *ego_vehicle_velocity,ego_vehicle_car_angle)

    filled_safety_box_og = fill_safety_box_with_heatmap_values(safety_box_array_og, heatmap,time)

    # Convert to DataFrame for visualization (optional)
    filled_safety_box_og_df = pd.DataFrame(filled_safety_box_og)

    time_series_og.append(np.rot90(filled_safety_box_og))



    #dynamical safety box
    safety_box_corners_large, safety_box_array, safety_box_df = fill_safety_box_with_occupancy(heatmap, *ego_vehicle_position, *ego_vehicle_velocity,ego_vehicle_car_angle)


    # maximum dynamical safety box  # Max safety_box based on the max velocity within the planned trajectory
    ego_vehicle_velocity_max = (np.max(abs(df_av1["velocity_x"])), np.max(abs(df_av1["velocity_y"])))
    #safety_box_corners_large_max, safety_box_array_max, safety_box_df_max,safety_box_array_distance_max, safety_box_df_distance_max = fill_safety_box_with_occupancy(heatmap, *ego_vehicle_position, *ego_vehicle_velocity_max,ego_vehicle_car_angle)
    safety_box_corners_large_max, safety_box_array_max, safety_box_df_max = fill_safety_box_with_occupancy(heatmap, *ego_vehicle_position, *ego_vehicle_velocity_max,ego_vehicle_car_angle)

    
    # add the maximum for now
    filled_safety_box = fill_safety_box_with_heatmap_values(safety_box_array_max, heatmap,time)
    # Convert to DataFrame for visualization (optional)
    filled_safety_box_df = pd.DataFrame(filled_safety_box)

    time_series.append(np.rot90(filled_safety_box))


    # PLOT START HERE

    # Example usage
    t = time

    car_position_av = (df_av1.iloc[int(t)]["position_x"], df_av1.iloc[int(t)]["position_y"]) #x,y
    car_velocity_av = (df_av1.iloc[int(t)]["velocity_x"], df_av1.iloc[int(t)]["velocity_y"]) #x,y
    car_angle_av = np.arctan2(df_av1.iloc[int(t)]["velocity_y"], df_av1.iloc[int(t)]["velocity_x"])# df_av1.iloc[int(time)]["heading"]
    box_length = lengths #lengths[df_av1.iloc[0]["object_type"]]
    box_width = widths #widths[df_av1.iloc[0]["object_type"]]

    fig, ax = plt.subplots(figsize=(8, 6))

    safety_box_corners_av = calculate_safety_box(*car_position_av, *car_velocity_av, car_angle_av,box_length,box_width)
    #plt.plot(safety_box_corners[:, 0], safety_box_corners[:, 1], '-o',color='pink')  # Plot the corners
    #plt.scatter(car_position[0],car_position[1], color='red')  # Mark the car's position

    ax.plot(safety_box_corners_av[:, 0], safety_box_corners_av[:, 1], '-o', color="magenta")
    ax.plot(safety_box_corners_large_og[:, 0], safety_box_corners_large_og[:, 1], '-o', color="blue")

    dynamic_safety_box_all = safety_box_array[:,int((-safety_box_array_og.shape[1]+safety_box_array.shape[1])/2):]
    dynamic_safety_box_all_large = np.array([dynamic_safety_box_all[0][0], dynamic_safety_box_all[-1][0], dynamic_safety_box_all[-1][-1], dynamic_safety_box_all[0][-1],dynamic_safety_box_all[0][0]])
    # Need to check if it applies for all cases..
    ax.plot(dynamic_safety_box_all_large[:, 0], dynamic_safety_box_all_large[:, 1], '-o', color="cyan")
    ax.scatter(car_position_av[0], car_position_av[1], color='black')  # Mark the car's position

    # Set labels, title, limits, and grid
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.set_title('Safety Box around the Car')
    # ax.set_xlim(minx, maxx)
    # ax.set_ylim(miny, maxy)
    ax.grid(True)
    #plt.axis('equal')
    # ax = plt.gca()  # Get current axis
    # ax.set_aspect('equal', adjustable='box')
    


    ##### ACTUAL SAFETY BOX AT EACH TIME STEP####
    heatmap_df = create_heatmap_dataframe(minx, maxx, miny, maxy, safety_box_corners_large) #(minx,maxx,miny,maxy)
    #print(heatmap_df)
    x_coords = []
    y_coords = []
    x_coords_check = []
    y_coords_check = []
    weights = []
    checks = []

    #filled_safety_box_df is based on the maximum
    #actual_heat_map_safety_box = safety_box_array_max[:,int((-safety_box_array_og.shape[1]+safety_box_array_max.shape[1])/2): int((-safety_box_array_max.shape[1]+safety_box_array.shape[1])/2)]
    
    actual_heat_map_safety_box = safety_box_array_max[:,int((-safety_box_array_og.shape[1]+safety_box_array_max.shape[1])/2): int((-safety_box_array_max.shape[1]+safety_box_array.shape[1])/2)-1]
    actual_heat_map_safety_box_df = pd.DataFrame(actual_heat_map_safety_box)
    # add the maximum for now

    print(int((-safety_box_array_og.shape[1]+safety_box_array_max.shape[1])/2))
    print(int((-safety_box_array_max.shape[1]+safety_box_array.shape[1])/2))
    print(actual_heat_map_safety_box)
    filled_safety_box_actual = fill_safety_box_with_heatmap_values(actual_heat_map_safety_box, heatmap,time)

    # Convert to DataFrame for visualization (optional)
    filled_safety_box_actual_df = pd.DataFrame(filled_safety_box_actual)

    time_series_each.append(np.rot90(filled_safety_box_actual))

    for col in actual_heat_map_safety_box_df.columns:
        for i in range(len(actual_heat_map_safety_box_df[col])):
            cell = actual_heat_map_safety_box_df[col][i]
            #if filled_safety_box_df[col][i] !=0:
            # Safely evaluate the string as a list
            x_coords.append(cell[0])
            y_coords.append(cell[1])
            weights.append(filled_safety_box_actual_df[col][i])
            if filled_safety_box_actual_df[col][i] !=0:
              x_coords_check.append(cell[0])
              y_coords_check.append(cell[1])
              checks.append(filled_safety_box_actual_df[col][i])

    cmap = plt.cm.RdYlGn_r

    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable


    # Define the normalization based on the range of values from all datasets
    norm = Normalize(vmin=0, vmax=1)

    # Create a ScalarMappable for consistent color mapping
    sm = ScalarMappable(norm=norm, cmap=cmap)

      # Use subplots to explicitly create a figure and axes
    sc = ax.scatter(x_coords, y_coords, c=sm.to_rgba(weights), alpha=0.3)
    # Create a colorbar with a reference to the Axes and the ScalarMappable


    #print(np.max(weights))

    #print(np.min(weights))
    #print(len(checks))

    plt.colorbar(sm, ax=ax, label='Value')  # Explicitly pass the Axes object here

    plt.savefig('scenario'+str(round(t,1))+'.png')
    plt.close()

    plt.figure()
    plt.scatter(x_coords_check, y_coords_check, c=checks,cmap = cm.RdYlGn_r)
    plt.savefig('scenario_check'+str(round(t,1))+'.png')
    plt.close()
    # PLOT END HERE




35.95 -60.05 79.95 -5.05
0
(100, 100)
11.985315538192722
120
(100, 120)
33.79072413529735
338
(100, 338)
119
-109
[[array([ 65.15027892, -24.8214739 ]) array([ 65.08788586, -24.89996608])
  array([ 65.02549281, -24.97845825]) ...
  array([ 58.53661487, -33.14164417]) array([ 58.47422181, -33.22013634])
  array([ 58.41182875, -33.29862851])]
 [array([ 65.07120677, -24.75861982]) array([ 65.00881371, -24.837112  ])
  array([ 64.94642066, -24.91560417]) ...
  array([ 58.45754271, -33.07879009]) array([ 58.39514966, -33.15728226])
  array([ 58.3327566 , -33.23577443])]
 [array([ 64.99213462, -24.69576574]) array([ 64.92974156, -24.77425792])
  array([ 64.8673485 , -24.85275009]) ...
  array([ 58.37847056, -33.01593601]) array([ 58.31607751, -33.09442818])
  array([ 58.25368445, -33.17292035])]
 ...
 [array([ 57.48028022, -18.72462815]) array([ 57.41788717, -18.80312033])
  array([ 57.35549411, -18.8816125 ]) ...
  array([ 50.86661617, -27.04479842]) array([ 50.80422311, -27.12329059])
  ar

In [18]:
time_series = np.array(time_series)

lengths = 5
widths = 2
resolution = 0.1

Y, X = time_series[0].shape
center_x, center_y = X // 2, Y // 2  # center the ellipse within your grid
length_x,length_y =  (2*widths)*(1/resolution),(lengths+widths)*(1/resolution)

start_x = center_x - int(length_x // 2)
end_x = center_x + int(length_x // 2)
start_y = center_y - int(length_y // 2)
end_y = center_y + int(length_y // 2)

# Create a 3D mask with the same shape as data_3d
mask_small = np.zeros((Y, X), dtype=bool)

# Set values inside the rectangle to True
mask_small[start_y:end_y, start_x:end_x] = True

risk_data_3d = np.zeros_like(time_series[0])
risk_data_3d[mask_small] = 1




# range_end = ((2*lengths) - (lengths+widths))*(1/resolution) 
range_end = int(len(time_series[0]))+1

# for i in range(0,int(range_end),1):

for i in range(0,range_end,1):

    length_x,length_y = (2*widths)*(1/resolution)+i,(lengths+widths)*(1/resolution)+i
    start_x = center_x - int(length_x // 2)
    end_x = center_x + int(length_x // 2)
    start_y = center_y - int(length_y // 2)
    end_y = center_y + int(length_y // 2)

    # Create a 3D mask with the same shape as data_3d
    mask_test_lower = np.zeros((Y, X), dtype=bool)

    # Set values inside the rectangle to True
    mask_test_lower[start_y:end_y, start_x:end_x] = True

    length_x,length_y = (2*widths)*(1/resolution)+i+1,(lengths+widths)*(1/resolution)+i+1
    start_x = center_x - int(length_x // 2)
    end_x = center_x + int(length_x // 2)
    start_y = center_y - int(length_y // 2)
    end_y = center_y + int(length_y // 2)

    # Create a 3D mask with the same shape as data_3d
    mask_test_upper = np.zeros((Y, X), dtype=bool)

    # Set values inside the rectangle to True
    mask_test_upper[start_y:end_y, start_x:end_x] = True


    risk_data_3d[np.logical_and(mask_test_upper, ~mask_test_lower)] = 1-i/range_end

risk_data_3d = risk_data_3d.astype(float) #heatmap risk

reversed_time_series=time_series[:,::-1, :] # locational based risk



# case 5 exponential method (Cox Model)
# risk_update = risk_basic at time t = 0
# Then it get updated after time t= 0

percent_change = []
for i in range(len(reversed_time_series)):
    if i ==0:
        row = np.ones_like(reversed_time_series[i]) # zero risk of collision
        percent_change.append(row)
    else:
        row = np.exp((reversed_time_series[i]-reversed_time_series[i-1]).astype(float))
        #row = np.multiply(row, percent_change[-1])
        #row = np.minimum(row,1)
        percent_change.append(row)

percent_change = np.array(percent_change)

risk_max_no_boost = []
risk_time_series =[]
vmax = 0

risk_max = []
risk_sum = []

risk_time_series_norm =[]

risk_max_norm = []
risk_sum_norm = []


a = 1/(10*10)







max_boost = np.exp(1)
for t in range(len(reversed_time_series)):

    result_temp = reversed_time_series[t]*risk_data_3d
    risk_max_no_boost.append(np.max(result_temp))

    result_temp = reversed_time_series[t]*risk_data_3d*percent_change[t]


    ############ FILTER only the values within the safety box #############
  
    ####################### HEREHREHREHRHERHEREH##################
    # Here we need to filter based on the index here:

    # This is the actual safety box we saw 

    ego_vehicle_position = (df_av1.iloc[int(t)]["position_x"], df_av1.iloc[int(t)]["position_y"])
    ego_vehicle_velocity = (df_av1.iloc[int(t)]["velocity_x"], df_av1.iloc[int(t)]["velocity_y"])
    ego_vehicle_car_angle = np.arctan2(df_av1.iloc[int(t)]["velocity_y"], df_av1.iloc[int(t)]["velocity_x"])# df_av1.iloc[int(time)]["heading"]

    heatmap = load_heatmap(t)  # Access preloaded heatmap
    #safety_box_corners_large, safety_box_array, safety_box_df,safety_box_array_distance, safety_box_df_distance = fill_safety_box_with_occupancy(heatmap, *ego_vehicle_position, *ego_vehicle_velocity,ego_vehicle_car_angle)

    # original safety box
    safety_box_corners_large_og, safety_box_array_og, safety_box_df_og = fill_safety_box_with_occupancy_og(heatmap, *ego_vehicle_position, *ego_vehicle_velocity,ego_vehicle_car_angle)


    safety_box_corners_large, safety_box_array, safety_box_df = fill_safety_box_with_occupancy(heatmap, *ego_vehicle_position, *ego_vehicle_velocity,ego_vehicle_car_angle)
    
    # maximum dynamical safety box  # Max safety_box based on the max velocity within the planned trajectory
    ego_vehicle_velocity_max = (np.max(abs(df_av1["velocity_x"])), np.max(abs(df_av1["velocity_y"])))
    #safety_box_corners_large_max, safety_box_array_max, safety_box_df_max,safety_box_array_distance_max, safety_box_df_distance_max = fill_safety_box_with_occupancy(heatmap, *ego_vehicle_position, *ego_vehicle_velocity_max,ego_vehicle_car_angle)
    safety_box_corners_large_max, safety_box_array_max, safety_box_df_max = fill_safety_box_with_occupancy(heatmap, *ego_vehicle_position, *ego_vehicle_velocity_max,ego_vehicle_car_angle)

    

    ########################## Index based #################################

    result_temp_filter = result_temp[int((-safety_box_array_og.shape[1]+safety_box_array_max.shape[1])/2): int((-safety_box_array_max.shape[1]+safety_box_array.shape[1])/2)-1, :]



    ########################################################################

    #result = np.minimum(result_temp,1)
    if vmax <= np.max(result_temp_filter):
        vmax = np.max(result_temp_filter)
    #risk_time_series.append(result_temp)
    risk_max.append(np.max(result_temp_filter))
    #risk_sum.append(1-np.exp(-1*np.sum(result_temp)*a))

    #risk_time_series_norm.append(result_temp/max_boost)
    risk_max_norm.append(np.max(result_temp_filter/max_boost))
    print(np.max(result_temp_filter/max_boost))
    #risk_sum_norm.append(1-np.exp(-1*np.sum(result_temp/max_boost)*a))


(100, 100)
11.985315538192722
120
(100, 120)
33.79072413529735
338
(100, 338)
0.0735310264758872
(100, 100)
12.165318486700079
122
(100, 122)
33.79072413529735
338
(100, 338)
0.06171393819665207
(100, 100)
12.353613885764991
124
(100, 124)
33.79072413529735
338
(100, 338)
0.05602157481316786
(100, 100)
12.580630792789666
126
(100, 126)
33.79072413529735
338
(100, 338)
0.052278772320518994
(100, 100)
12.792974638739892
128
(100, 128)
33.79072413529735
338
(100, 338)
0.0475160071452942
(100, 100)
13.024218357295899
131
(100, 131)
33.79072413529735
338
(100, 338)
0.04944807780183067
(100, 100)
13.24326180306953
133
(100, 133)
33.79072413529735
338
(100, 338)
0.04174624042688612
(100, 100)
13.48882852441624
135
(100, 135)
33.79072413529735
338
(100, 338)
0.04201930568763042
(100, 100)
13.766457026380767
138
(100, 138)
33.79072413529735
338
(100, 338)
0.037385590635049065
(100, 100)
14.011703120676462
141
(100, 141)
33.79072413529735
338
(100, 338)
0.03564937814060619
(100, 100)
14.18972019

In [19]:
plt.figure(figsize=(8, 5))
plt.plot(range(len(risk_max_norm)),np.array(risk_max_norm))
plt.savefig("PORA_Ex.png", dpi=300)

In [20]:
for i in range(len(risk_max_norm)):
    print(i)
    print(risk_max_norm[i])

0
0.0735310264758872
1
0.06171393819665207
2
0.05602157481316786
3
0.052278772320518994
4
0.0475160071452942
5
0.04944807780183067
6
0.04174624042688612
7
0.04201930568763042
8
0.037385590635049065
9
0.03564937814060619
10
0.03252959234286617
11
0.03061406921863042
12
0.03297797115795401
13
0.030010855933364174
14
0.027686554470858774
15
0.024773868754689087
16
0.025583180726947347
17
0.024379498303545355
18
0.01665654643251834
19
0.019239848303128197
20
0.012613412617635484
21
0.010779882875276255
22
0.009810980681260684
23
0.006009423988768867
24
0.003381431987486102
25
0.0015551640330137696
26
0.0011319562863867
27
0.000787566915355677
28
0.00043294357552467396
29
0.0002190711611091554
30
0.00030131238718856537
31
0.00013995135213226438
32
0.00011526037055359184
33
5.68378298924299e-05
34
1.745334237272786e-05
35
1.0375848991836627e-05
36
3.322833162634641e-06
37
2.016267947946965e-06
38
2.3152949813361056e-06
39
1.0470760066122467e-06
40
5.792817182933955e-07
41
3.739613365913857e-